# Step-matched inference at 4e6 — 260914 vs 260723

Runs self-play inference on models taken at **~4e6 steps**, so the two batches can be compared at
the same amount of training.

**Why.** 260914 central models choose the High wall on almost every trial (`p_high` ~1.00) where
260723 central models choose it about 80% of the time. The tournament between them is confounded:
260914 finishes at 7e6 steps and 260723 at 3e6. Taking both at ~4e6 removes the training-length part
of that gap.

**Why more central candidates.** Centrality at 4e6 is not the centrality at the finished policy, and
does not move predictably. Of the first two entrants, `260914_0020` went 0.870 -> 0.890 but
`260914_0014` went 0.880 -> 0.784, which drops it below its batch's main band of 0.81-0.91. Picking
a field whose members are centrality-matched **at the step actually being raced** means simulating
several candidates at 4e6 and choosing afterwards. `EXTRA_CENTRAL` below holds the next most typical
central models for that.

**How a specific checkpoint gets used.** `mlagents-learn --resume` always loads
`<run>/<behaviour>/checkpoint.pt`, so there is no flag for "run at step N". The staging cell copies
the per-step `OctagonAgentSocial-<step>.pt` into a throwaway run dir as `checkpoint.pt`, the same
trick `tournament_func.assemble_matchup_run_dir` uses. Source run dirs are only ever read.

**Each batch runs under its own training build and config**, matching how its existing sims were
made, so the numbers stay comparable to those already measured for that batch.

**Re-running is safe.** Staging skips a model that is already staged. The status cell classifies
every candidate as done, incomplete or not started, and only the ones that are not done are handed
to `batch_inference`. Nothing finished is ever recomputed or overwritten. Incomplete folders are
moved aside rather than deleted, because a partial JSON left beside a fresh one gives two `.json`
files in the folder, which `octagon_analysis` rejects when loading.

**Before running:** the kernel must be the `mlagents` env, and CUDA should be healthy
(`nvidia-smi`, `torch.cuda.is_available()`) per the driver notes in `CLAUDE.md`. Port 7000 is used,
clear of training at 5005+20*i.

Analysis of the resulting logs happens in `octagon_analysis`, not here.

In [6]:
import re
import shutil
import sys
from datetime import datetime
from pathlib import Path

REPO_ROOT = Path("~/repos/agent_training").expanduser()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from batch_inference_func import batch_inference

OCTAGON = Path("~/Unity/Octagon").expanduser()
TARGET_STEP = 4_000_000
BEH = "OctagonAgentSocial"

# throwaway run dirs holding the 4e6 checkpoints, one folder per batch
STAGE = OCTAGON / "results" / "stepmatch_4e6"

# where the sim JSONs land, one folder per batch so each reads as its own batch in octagon_analysis
SIM_SUBDIR = "stepmatch_4e6"
SIM_ROOT = OCTAGON / "simulations" / SIM_SUBDIR

# interrupted runs are moved here rather than deleted
SUPERSEDED = OCTAGON / "simulations" / "superseded" / SIM_SUBDIR

## Candidates

`EXTRA_CENTRAL` is the list to edit. It holds 260914 central models ranked next by typicality after
the two already run, taken from the ranking in
`octagon_analysis/demos/strategy_characterisation/models/competitive/model_characterisation_demo260914.ipynb`.
Add or remove ids freely: anything already simulated is skipped, so the cost of an extra id is one
sim and the cost of re-running the notebook is nothing.

In [7]:
# the two entrants already simulated at 4e6, plus the off-cluster model
DONE_ALREADY = ["260914_0020", "260914_0014", "260914_0017"]

# more 260914 central models, so the field can be centrality-matched at 4e6 rather than at 7e6.
# ranked by typicality within the central cluster: 20 and 14 were 1st and 2nd, these come next
EXTRA_CENTRAL = ["260914_0013", "260914_0023", "260914_0007",
                 "260914_0012", "260914_0005", "260914_0021"]

# each batch keeps its own training build and config
JOBS = {
    "260914": dict(
        source=OCTAGON / "results" / "2609" / "260914",
        build=OCTAGON / "builds" / "260914" / "build_260914_01_social.x86_64",
        config=OCTAGON / "Assets/Scripts/MLConfigFiles/SocialConfig260914.yaml",
        run_ids=DONE_ALREADY + EXTRA_CENTRAL,
    ),
    "260723": dict(
        source=OCTAGON / "results" / "2607" / "260723",
        build=OCTAGON / "builds" / "260723" / "build_260723_01_social.x86_64",
        config=OCTAGON / "Assets/Scripts/MLConfigFiles/SocialConfig.yaml",
        run_ids=["260723_0034", "260723_0039"],
    ),
}

TRIAL_SEQ = REPO_ROOT / "trial_sequences" / "trials_100000_seed17.json"
EPISODES = 201          # ~200 trials, matching the existing sims for both batches
BASE_PORT = 7000        # training holds 5005-5021, so stay clear

for batch, job in JOBS.items():
    print(f"{batch}: {len(job['run_ids'])} candidates -> {job['run_ids']}")

260914: 9 candidates -> ['260914_0020', '260914_0014', '260914_0017', '260914_0013', '260914_0023', '260914_0007', '260914_0012', '260914_0005', '260914_0021']
260723: 2 candidates -> ['260723_0034', '260723_0039']


## Stage the 4e6 checkpoints

Copies the per-step checkpoint nearest `TARGET_STEP` into `STAGE/<batch>/<run_id>/`, with the run's
`configuration.yaml` beside it. Those two files are all inference reads.

A model already staged is left alone, so this never half-replaces a checkpoint and never disturbs a
model whose sim has already run. Delete `STAGE/<batch>/<run_id>` to force one to be re-staged.

In [8]:
def nearest_checkpoint(run_dir, target=TARGET_STEP):
    """ Return the (step, path) of the per-step checkpoint nearest target. Takes a run dir. """

    cands = []
    for p in (run_dir / BEH).glob(f"{BEH}-*.pt"):
        m = re.search(r"-(\d+)\.pt$", p.name)
        if m:
            cands.append((int(m.group(1)), p))
    if not cands:
        raise FileNotFoundError(f"no per-step checkpoints under {run_dir / BEH}")
    return min(cands, key=lambda t: abs(t[0] - target))


for batch, job in JOBS.items():
    for rid in job["run_ids"]:
        dst_run = STAGE / batch / rid
        if (dst_run / BEH / "checkpoint.pt").exists():
            print(f"{batch}/{rid}: already staged, left alone")
            continue
        step, src_ckpt = nearest_checkpoint(job["source"] / rid)
        (dst_run / BEH).mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_ckpt, dst_run / BEH / "checkpoint.pt")
        shutil.copy2(job["source"] / rid / "configuration.yaml", dst_run / "configuration.yaml")
        print(f"{batch}/{rid}: staged step {step:,}  (from {src_ckpt.name})")

260914/260914_0020: already staged, left alone
260914/260914_0014: already staged, left alone
260914/260914_0017: already staged, left alone
260914/260914_0013: staged step 4,024,826  (from OctagonAgentSocial-4024826.pt)
260914/260914_0023: staged step 4,024,786  (from OctagonAgentSocial-4024786.pt)
260914/260914_0007: staged step 4,024,915  (from OctagonAgentSocial-4024915.pt)
260914/260914_0012: staged step 4,024,639  (from OctagonAgentSocial-4024639.pt)
260914/260914_0005: staged step 4,024,888  (from OctagonAgentSocial-4024888.pt)
260914/260914_0021: staged step 4,024,915  (from OctagonAgentSocial-4024915.pt)
260723/260723_0034: already staged, left alone
260723/260723_0039: already staged, left alone


In [9]:
# read back the step each staged checkpoint actually holds, so a wrong file is caught before
# an hour of inference rather than after
import torch


def checkpoint_step(ckpt_path):
    """ Return the training step a checkpoint was saved at. Takes the checkpoint path.

        mlagents stores this under a name-mangled key inside global_step, so the lookup is
        guarded: a None means the step could not be read, not that it matches. """
    gs = torch.load(ckpt_path, map_location="cpu", weights_only=False).get("global_step")
    if hasattr(gs, "values"):
        gs = next(iter(gs.values()))
    return int(gs.item()) if hasattr(gs, "item") else None


steps = {}
for batch, job in JOBS.items():
    for rid in job["run_ids"]:
        steps[rid] = checkpoint_step(STAGE / batch / rid / BEH / "checkpoint.pt")
        print(f"{rid}  step {steps[rid]:,}" if steps[rid] else f"{rid}  step UNREADABLE")

got = [s for s in steps.values() if s]
if len(got) == len(steps):
    print(f"\nspread {max(got) - min(got):,} steps "
          f"({100 * (max(got) - min(got)) / max(got):.2f}% of the largest)")
else:
    print("\nWARNING: at least one step could not be read")

260914_0020  step 4,024,887
260914_0014  step 4,024,831
260914_0017  step 4,024,942
260914_0013  step 4,024,826
260914_0023  step 4,024,786
260914_0007  step 4,024,915
260914_0012  step 4,024,639
260914_0005  step 4,024,888
260914_0021  step 4,024,915
260723_0034  step 4,000,042
260723_0039  step 3,999,960

spread 24,982 steps (0.62% of the largest)


## What still needs running

Every candidate falls into one of three states.

- **done** — `DONE.txt` present and exactly one `.json`. Never re-run, never touched.
- **incomplete** — a folder exists but the sim did not finish, usually an interrupted run. Its
  partial JSON has to be moved aside before re-running, or the retry leaves two `.json` files in the
  folder and `get_model_json_filename` raises on load.
- **not started** — no folder yet.

This cell only reports. The next one does the moving.

In [10]:
TRIAL_END = '"trial end"'          # counted in the raw log text, so a large file is never parsed


def sim_status(batch, rid):
    """ Return 'done', 'incomplete' or 'not started' for one model's sim folder.
        Takes a batch name and run id. """

    d = SIM_ROOT / batch / rid
    if not d.exists():
        return "not started"
    jsons = [p for p in d.glob("*.json") if not p.name.startswith("._")]
    if (d / "DONE.txt").exists() and len(jsons) == 1:
        return "done"
    return "incomplete"


def trial_end_count(batch, rid):
    """ Return how many trials the logged sim completed, or 0 if there is no single log.
        Takes a batch name and run id. """

    jsons = [p for p in (SIM_ROOT / batch / rid).glob("*.json") if not p.name.startswith("._")]
    if len(jsons) != 1:
        return 0
    return jsons[0].read_text(encoding="utf-8-sig", errors="ignore").count(TRIAL_END)


status = {(b, r): sim_status(b, r) for b, job in JOBS.items() for r in job["run_ids"]}

for (b, r), s in status.items():
    extra = f"  trial_ends={trial_end_count(b, r)}" if s == "done" else ""
    print(f"{b}/{r}: {s}{extra}")

counts = {s: sum(1 for v in status.values() if v == s) for s in ("done", "incomplete", "not started")}
print(f"\ndone {counts['done']}   incomplete {counts['incomplete']}   "
      f"not started {counts['not started']}")
todo = counts["incomplete"] + counts["not started"]
print(f"to run this pass: {todo} sims, roughly {0.4 * todo:.1f} h at ~25 min each")

260914/260914_0020: done  trial_ends=199
260914/260914_0014: done  trial_ends=199
260914/260914_0017: done  trial_ends=199
260914/260914_0013: not started
260914/260914_0023: not started
260914/260914_0007: not started
260914/260914_0012: not started
260914/260914_0005: not started
260914/260914_0021: not started
260723/260723_0034: done  trial_ends=199
260723/260723_0039: done  trial_ends=199

done 5   incomplete 0   not started 6
to run this pass: 6 sims, roughly 2.4 h at ~25 min each


## Move aside any incomplete run

Moves each incomplete folder to `simulations/superseded/stepmatch_4e6/<batch>/<run_id>__<timestamp>`
so the retry starts clean. Nothing is deleted, and folders marked **done** are never touched.

If the status above shows no incomplete runs, this cell does nothing.

In [11]:
moved = 0
for (batch, rid), s in status.items():
    if s != "incomplete":
        continue
    src = SIM_ROOT / batch / rid
    dst = SUPERSEDED / batch / f"{rid}__{datetime.now():%Y%m%d_%H%M%S}"
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dst))
    print(f"{batch}/{rid}: moved aside -> {dst}")
    moved += 1

print(f"\n{moved} incomplete run(s) moved aside" if moved else "\nnothing to move aside")

# recompute, so the run cell below sees the cleaned state
status = {(b, r): sim_status(b, r) for b, job in JOBS.items() for r in job["run_ids"]}


nothing to move aside


## Run the inference sims

One `batch_inference` call per batch, so each uses its own build and config. Only models not already
done are passed in, so this cell can be re-run as often as needed and will only ever pick up what is
missing. Sims run sequentially at roughly 17-30 min each depending on what else is on the GPU.

If it is interrupted again, just re-run the status cell, the move-aside cell and this one.

In [12]:
logs = {}
for batch, job in JOBS.items():
    pending = [r for r in job["run_ids"] if status[(batch, r)] != "done"]
    if not pending:
        print(f"\n{'#' * 20} {batch}: nothing pending {'#' * 20}", flush=True)
        continue
    print(f"\n{'#' * 20} {batch}: running {pending} {'#' * 20}", flush=True)
    logs[batch] = batch_inference(
        run_dir=OCTAGON,
        unity_env_path=job["build"],
        patched_yaml_path=job["config"],
        out_root=OCTAGON / "simulations",
        results_subdir=f"{SIM_SUBDIR}/{batch}",
        run_ids=pending,
        trial_seq=TRIAL_SEQ,
        episodes=EPISODES,
        timeout_s=5000,
        seed=17,
        base_port=BASE_PORT,
    )


#################### 260914: running ['260914_0013', '260914_0023', '260914_0007', '260914_0012', '260914_0005', '260914_0021'] ####################
Found 6 models to run inference on: ['260914_0013', '260914_0023', '260914_0007', '260914_0012', '260914_0005', '260914_0021']

=== [1/6] Running inference: 260914_0013 ===
['mlagents-learn', '/home/tom/Unity/Octagon/Assets/Scripts/MLConfigFiles/SocialConfig260914.yaml', '--run-id', 'stepmatch_4e6/260914/260914_0013', '--resume', '--inference', '--base-port', '7000', '--env', '/home/tom/Unity/Octagon/builds/260914/build_260914_01_social.x86_64', '--no-graphics', '--env-args', '--sim_out', '/home/tom/Unity/Octagon/simulations/stepmatch_4e6/260914/260914_0013', '--sim_eps', '201', '--trial_seq', '/home/tom/repos/agent_training/trial_sequences/trials_100000_seed17.json', '--seed', '17']
=== Finished: 260914_0013 -> /home/tom/Unity/Octagon/simulations/stepmatch_4e6/260914/260914_0013/mlagents_stdout.log ===

=== [2/6] Running inference: 26091

## Check what finished

Trial counts should land near `EPISODES - 1`. A much lower count means the sim stalled or was cut
short, in which case delete that model's `DONE.txt` and re-run from the status cell.

In [13]:
for batch, job in JOBS.items():
    for rid in job["run_ids"]:
        s = sim_status(batch, rid)
        n_trials = trial_end_count(batch, rid) if s == "done" else 0
        flag = "" if (s == "done" and n_trials > 0.9 * (EPISODES - 1)) else "   <-- CHECK"
        print(f"{batch}/{rid}: {s:<12} trial_ends={n_trials}{flag}")

260914/260914_0020: done         trial_ends=199
260914/260914_0014: done         trial_ends=199
260914/260914_0017: done         trial_ends=199
260914/260914_0013: done         trial_ends=199
260914/260914_0023: done         trial_ends=199
260914/260914_0007: done         trial_ends=199
260914/260914_0012: done         trial_ends=199
260914/260914_0005: done         trial_ends=199
260914/260914_0021: done         trial_ends=199
260723/260723_0034: done         trial_ends=199
260723/260723_0039: done         trial_ends=199


## Next

The logs land in `~/Unity/Octagon/simulations/stepmatch_4e6/<batch>/<run_id>/`, one batch per
folder, which `parse_data.identify_sim_filepaths.get_model_folders` reads as a normal batch.

Centrality and `p_high` at 4e6 then come from `octagon_analysis`, pointing `mc.collect_model_data`
at each batch folder. Pick the field from the candidates whose **4e6** centrality clusters together,
rather than from the finished-policy ranking. For reference, the two already measured at 4e6 are
`260914_0020` at 0.890 and `260914_0014` at 0.784, against a finished-policy band of 0.81-0.91.

Then update the entrants in `agent_training/tournament/run_tournament_260914.ipynb`.